In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import  OneHotEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [7]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test= train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [10]:
x_train

,age,gender,fever,cough,city
20,12,Male,98.0,Strong,Bangalore
91,38,Male,NaN,Mild,Delhi
26,19,Female,100.0,Mild,Kolkata
7,20,Female,NaN,Strong,Mumbai
25,23,Male,NaN,Mild,Mumbai
...,...,...,...,...,...
85,16,Female,103.0,Mild,Bangalore
12,25,Female,99.0,Strong,Kolkata
84,69,Female,98.0,Strong,Mumbai
18,64,Female,98.0,Mild,Bangalore


# Normal transformer #

In [12]:
si = SimpleImputer()
x_train_fever = si.fit_transform(x_train[['fever']])

x_test_fever = si.fit_transform(x_test[['fever']])

x_train_fever.shape

(80, 1)

In [14]:
# Ordinal Encoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough = oe.fit_transform(x_train[['cough']])
x_test_cough = oe.fit_transform(x_test[['cough']])
x_train_cough.shape

(80, 1)

In [18]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])

# also the test data
x_test_gender_city = ohe.fit_transform(x_test[['gender','city']])

x_train_gender_city.shape

(80, 4)

In [20]:

# Extracting Age
x_train_age = x_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
x_test_age = x_test.drop(columns=['gender','fever','cough','city']).values

x_train_age.shape

(80, 1)

In [21]:
X_train_transformed = np.concatenate((x_train_age,x_train_fever,x_train_gender_city,x_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

# Perform Column Transformation using Sklearn #

In [22]:
from sklearn.compose import ColumnTransformer

In [26]:
transformer = ColumnTransformer(transformers=[('tnf1',SimpleImputer(),['fever']),('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])],remainder='passthrough')

In [29]:
transformer.fit_transform(x_train).shape

(80, 7)

In [30]:
transformer.transform(x_test).shape

(20, 7)